# Peak Finding Example

This notebook demonstrates how to locate and characterise gamma-ray photopeaks in a
measured spectrum using the tools provided in `gs_analysis.py`.

Two complementary peak-finding approaches are shown:

| Method | Function | Description |
|---|---|---|
| Scipy prominence | `peak_finder` | Applies 5-point smoothing then uses `scipy.signal.find_peaks` with a prominence threshold |
| Mariscotti 2nd-difference | `mariscotti_peak_finder` | Applies smoothing then identifies peaks from significant negative values in the second difference of the spectrum |

The example uses a real Co-60 measurement (`Co_60_raised_1.Spe`) from the `test_data/`
directory. Co-60 has two strong photopeaks at **1173.2 keV** and **1332.5 keV**, which
makes it a convenient calibration source for demonstrating peak detection.

## 1. Setup – import modules

In [ ]:
import sys
import os

# Add the package root to sys.path when running from the examples/ directory
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt

import gs_spe_reading
import gs_analysis

## 2. Load the spectrum

Read the Co-60 `.Spe` file (Maestro ASCII format) and generate the energy-bin axis
from the calibration coefficients stored in the file.

In [ ]:
# Path to the Co-60 spectrum relative to the examples/ directory
SPE_PATH = os.path.join('..', 'test_data', 'Co_60_raised_1.Spe')

spec = gs_spe_reading.read_dollar_spe(SPE_PATH)
ebins = gs_analysis.generate_ebins(spec)  # keV per channel

print(f"Channels   : {spec.num_channels}")
print(f"Live time  : {spec.live_time:.1f} s")
print(f"Total counts: {spec.counts.sum():,}")
print(f"Energy range: {ebins[0]:.1f} - {ebins[-1]:.1f} keV")

## 3. Plot the raw spectrum

A logarithmic y-axis is used so that both low- and high-count regions are visible
simultaneously.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.step(ebins, spec.counts, linewidth=0.7, color='steelblue', label='Co-60 raw spectrum')
ax.set_yscale('log')
ax.set_xlabel('Energy (keV)', fontsize=12)
ax.set_ylabel('Counts', fontsize=12)
ax.set_title('Co-60 Raw Spectrum', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Smooth the spectrum

Both peak-finding methods operate on a smoothed version of the spectrum. The
`five_point_smooth` function applies the 5-point weighted average recommended by
G.W. Phillips (Nucl. Instrum. Methods 153, 1978). Applying it twice reduces noise
further while preserving the peak shape.

In [ ]:
# Two passes of 5-point smoothing
smoothed = gs_analysis.five_point_smooth(spec.counts)
smoothed = gs_analysis.five_point_smooth(smoothed)

fig, ax = plt.subplots(figsize=(10, 5))
ax.step(ebins, spec.counts, linewidth=0.5, color='steelblue', alpha=0.5, label='Raw')
ax.plot(ebins, smoothed, linewidth=0.9, color='darkorange', label='Smoothed (2× 5-pt)')
ax.set_yscale('log')
ax.set_xlabel('Energy (keV)', fontsize=12)
ax.set_ylabel('Counts', fontsize=12)
ax.set_title('Raw vs Smoothed Co-60 Spectrum', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Peak finding – scipy prominence method

`gs_analysis.peak_finder` wraps `scipy.signal.find_peaks` and accepts two tuning
parameters:

- **`prominence`** – minimum vertical distance between a peak and its surrounding
  baseline. Increase to suppress weak peaks; decrease to detect more peaks.
- **`wlen`** – window length (channels) used when computing prominence. Smaller
  values make the algorithm more sensitive to narrow features.

In [ ]:
# Tune these parameters for your detector and source
PROMINENCE = 90   # minimum peak prominence in counts
WLEN       = 10   # window length in channels

smoothed_sp, peaks_sp = gs_analysis.peak_finder(spec.counts, PROMINENCE, WLEN)

print(f"Peaks found (channel index): {peaks_sp}")
print(f"Corresponding energies (keV): {ebins[peaks_sp].round(1)}")

In [ ]:
# Visualise the detected peaks on the spectrum
gs_analysis.plot_spect_peaks(smoothed_sp, ebins, peaks_sp)

## 6. Peak finding – Mariscotti 2nd-difference method

`gs_analysis.mariscotti_peak_finder` implements the method described in:
> M.A. Mariscotti, *Nuclear Instruments and Methods* **50** (1967) 309–320

The second difference of the smoothed spectrum is computed. Significant negative
values (below a threshold) mark positions where the curvature is large, i.e., peaks.

- **`threshold`** – negative threshold for the second difference. If `None`, it is
  set automatically from the statistics of the second-difference distribution.
- **`smooth_iterations`** – number of 5-point smoothing passes applied before the
  second difference is calculated (default 2).

In [ ]:
# Auto-threshold: set threshold=None to let the function choose automatically
smoothed_mar, peaks_mar = gs_analysis.mariscotti_peak_finder(
    spec.counts, threshold=None, smooth_iterations=2
)

print(f"Peaks found (channel index): {peaks_mar}")
print(f"Corresponding energies (keV): {ebins[peaks_mar].round(1)}")

In [ ]:
gs_analysis.plot_spect_peaks(smoothed_mar, ebins, peaks_mar)

## 7. Compare both methods

Plot both sets of detected peaks on the same axes so the results can be compared
directly.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(ebins, smoothed_sp, linewidth=0.8, color='steelblue', alpha=0.7, label='Smoothed spectrum')

# Scipy prominence peaks
ax.plot(ebins[peaks_sp], smoothed_sp[peaks_sp],
        'rv', markersize=8, label=f'Scipy prominence ({len(peaks_sp)} peaks)')

# Mariscotti peaks
ax.plot(ebins[peaks_mar], smoothed_mar[peaks_mar],
        'b^', markersize=8, label=f'Mariscotti 2nd-diff ({len(peaks_mar)} peaks)')

# Mark the known Co-60 lines for reference
for energy, label in [(1173.2, '1173.2 keV'), (1332.5, '1332.5 keV')]:
    ax.axvline(x=energy, color='green', linestyle='--', linewidth=1.0, alpha=0.6)
    ax.text(energy + 5, ax.get_ylim()[1] * 0.5, label,
            color='green', fontsize=8, rotation=90, va='top')

ax.set_yscale('log')
ax.set_xlabel('Energy (keV)', fontsize=12)
ax.set_ylabel('Counts', fontsize=12)
ax.set_title('Comparison of Peak-Finding Methods on Co-60 Spectrum', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Extract a peak region of interest (ROI)

`gs_analysis.get_peak_roi` extracts a symmetric window of channels centred on a
detected peak. The window half-width is set by the `offset` parameter.

Here we extract the ROI around the 1332.5 keV Co-60 photopeak.

In [ ]:
# Select the Co-60 1332.5 keV peak from the scipy results
# ebins[peaks_sp] was printed above — choose the index closest to 1332.5 keV
peak_energies_sp = ebins[peaks_sp]
co60_peak_idx = np.argmin(np.abs(peak_energies_sp - 1332.5))
peak_channel = peaks_sp[co60_peak_idx]

print(f"Selected peak: channel {peak_channel}, energy {ebins[peak_channel]:.1f} keV")

# Extract ±15 channels around the peak
roi_x, roi_y = gs_analysis.get_peak_roi(peak_channel, smoothed_sp, ebins, offset=15)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(roi_x, roi_y, s=12, color='steelblue', label='ROI counts')
ax.set_xlabel('Energy (keV)', fontsize=11)
ax.set_ylabel('Counts', fontsize=11)
ax.set_title('Peak ROI – Co-60 1332.5 keV photopeak', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 9. Fit a Gaussian to the peak

`gs_analysis.fit_peak` fits a Gaussian of the form

$$f(x) = A \exp\!\left(-\frac{(x - x_0)^2}{2\sigma^2}\right)$$

to the ROI data and returns the fitted parameters `[A, x0, sigma]`.

In [ ]:
params = gs_analysis.fit_peak(roi_x, roi_y)
amplitude, centroid, sigma = params

print(f"Gaussian fit parameters:")
print(f"  Amplitude (A)  : {amplitude:.1f} counts")
print(f"  Centroid (x0)  : {centroid:.2f} keV")
print(f"  Sigma (σ)      : {sigma:.2f} keV")
print(f"  FWHM           : {2.355 * abs(sigma):.2f} keV")

# Plot the ROI data with the fitted Gaussian overlaid
x_fit = np.linspace(roi_x[0], roi_x[-1], 300)
y_fit = gs_analysis.gaussian(x_fit, amplitude, centroid, sigma)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(roi_x, roi_y, s=12, color='steelblue', label='ROI counts', zorder=3)
ax.plot(x_fit, y_fit, color='crimson', linewidth=1.5,
        label=f'Gaussian fit  x₀={centroid:.1f} keV, FWHM={2.355*abs(sigma):.1f} keV')
ax.axvline(x=centroid, color='crimson', linestyle=':', linewidth=1.0, alpha=0.6)
ax.set_xlabel('Energy (keV)', fontsize=11)
ax.set_ylabel('Counts', fontsize=11)
ax.set_title('Gaussian Fit to Co-60 1332.5 keV Photopeak', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Calculate net counts

`gs_analysis.peak_counts` combines the ROI extraction and the trapezoid background
subtraction (`net_counts` with `m=1`) into a single call. It returns the channel
index of the peak and the estimated net counts (gross counts minus background).

The call below computes net counts for every peak detected by the scipy method.

In [ ]:
print(f"{'Peak #':<8} {'Channel':<10} {'Energy (keV)':<15} {'Net counts':<12}")
print("-" * 47)

for i in range(len(peaks_sp)):
    ch, net = gs_analysis.peak_counts(peaks_sp, i, smoothed_sp, ebins)
    print(f"{i:<8} {ch:<10} {ebins[ch]:<15.1f} {net:<12.0f}")

## Summary

This notebook demonstrated the complete peak-finding workflow:

1. **Load** a measured spectrum from a `.Spe` file.
2. **Smooth** the spectrum with the 5-point filter to reduce noise.
3. **Detect peaks** using either the scipy prominence method or the Mariscotti
   second-difference method.
4. **Visualise** the detected peaks on the spectrum.
5. **Extract** a peak region of interest (ROI) around a selected peak.
6. **Fit** a Gaussian to the ROI to measure the centroid energy and resolution (FWHM).
7. **Quantify** the peak area using background-subtracted net counts.

Both detection methods work well on clean spectra. The Mariscotti method can be more
sensitive to weak peaks in noisy spectra, while the scipy prominence method is easier
to tune via the single `prominence` parameter.